# Reading a protein into mBuild with its chemistry intact

A PDB file gives you element symbols and coordinates. To assign force
field parameters you need more than that: the bond orders and the formal
charge of every atom. Those are not in the file.

Most readers fill the gap by guessing — bonds from interatomic
distances, charges left at zero or copied from a column that is usually
blank. The guess is invisible until something downstream is wrong.

`mbuild.biopolymers` does not guess. Every residue is matched by atom
name against a template from the wwPDB Chemical Component Dictionary,
and the bonds, bond orders and formal charges come from that template. A
residue no template explains raises an error naming the residue.

In [ ]:
from mbuild.biopolymers import Protein

protein = Protein("1ubq_protonated.pdb")
print(len(list(protein.residues())), "residues,", protein.n_particles, "atoms")
print("net formal charge:", protein.net_formal_charge)

The charge is the part a coordinate reader cannot give you. It comes
from the matched templates, one residue at a time.

In [ ]:
for resnum in (1, 48, 76):
    residue = protein.get_residue(resnum, chain_id="A")
    print(f"{residue.name} {resnum:>3}  charge {residue.formal_charge:+d}  "
          f"{residue.atom_formal_charges}")

Bonds carry orders, so the structure is a chemical graph rather than a
set of connected points. `to_rdkit` is the shortest way to see that: it
refuses to export a bond whose order is unknown, so the fact that it
returns a sanitized molecule at all is the check.

In [ ]:
from rdkit import Chem

mol = protein.to_rdkit()
print(mol.GetNumAtoms(), "atoms, formal charge", Chem.GetFormalCharge(mol))

## Handing it to OpenFF

`save_pdb` writes a file with real residue numbers, chain identifiers,
a TER after each chain, and CONECT records for the bonds that residue
adjacency cannot imply — disulfides here. Peptide bonds are left
implied, because a residue-template reader rejects a CONECT its own
definitions cannot explain.

openff-pablo reads that file with no extra arguments.

In [ ]:
from openff.pablo import STD_CCD_CACHE, topology_from_pdb

protein.save_pdb("1ubq_prepared.pdb", overwrite=True)
topology = topology_from_pdb("1ubq_prepared.pdb", residue_library=STD_CCD_CACHE)

molecule = topology.molecule(0)
print(molecule.n_atoms, "atoms, net charge", molecule.total_charge)

In [ ]:
view = topology.visualize()
view.clear_representations()
view.add_representation("cartoon", color="#990000")
view

That is the round trip for an unmodified protein. The next notebook
does the part that needed new code: modifying the protein first, and
still handing OpenFF something it can read.